In [1]:
import numpy as np 
from pathlib import Path
from scipy.io import wavfile
import pandas as pd
import pickle
import librosa

# Make stimuli manifest for CommonVoice word recognition experimeent

Word recognition stimuli for human experiment run on prolific.

We will use excerpts from comonvoice evaluation set for all foregrounds. WOrds will be limited to the 998 word vocabulary in the set used for model training.

Pilot conditions:
* Clean speech
* Speech-shaped noise at 0dB
* Auditory Scene at 0dB 

Start with 1 example per word

### Curate test examples for human & model eval

Wants:
* 1 example per word. 
* words have valid 3 second window
* Gender balance 

In [2]:
## Import cv vocab and test manifest
commonvoice_path = Path('/om2/user/imgriff/datasets/commonvoice_9/en/')


vocab_fn = commonvoice_path / 'cv_word_int_label_dict.pkl'

with open(vocab_fn, 'rb') as handle:
    cv_vocab = pickle.load(handle)


cv_eval_manifest = pd.read_pickle(commonvoice_path / 'manifests/cv_word_rec_test_split_manifest.pdpkl')

In [3]:
cv_eval_manifest

,aud_path,client_id,gender,origin_index,word,start_in_s,end_in_s,sr,total_file_duration_in_s,dataset_splits,dataset_split_int,wsn_word_int,gender_int,client_int,word_examp_ix,cv_talker_int,cv_speaker_ident_split,cv_word_int
0,common_voice_en_20911758.mp3,0027fcb0b1be4cd41e38167393b6dff0ba22b28b4bb450...,male,16325,STAND,3.549125,3.930063,48000,6.240,test,2.0,NaN,1,608,5220,0,NaN,997
1,common_voice_en_20911758.mp3,0027fcb0b1be4cd41e38167393b6dff0ba22b28b4bb450...,male,16325,PRIVATE,2.346000,2.767063,48000,6.240,test,2.0,539.0,1,608,4741,0,NaN,419
2,common_voice_en_20911760.mp3,0027fcb0b1be4cd41e38167393b6dff0ba22b28b4bb450...,male,16327,HAD,1.424750,1.585250,48000,4.800,test,2.0,NaN,1,608,2228,0,NaN,38
3,common_voice_en_20911759.mp3,0027fcb0b1be4cd41e38167393b6dff0ba22b28b4bb450...,male,16326,GENERAL,2.524375,2.925063,48000,6.936,test,2.0,294.0,1,608,2152,0,NaN,268
4,common_voice_en_20911758.mp3,0027fcb0b1be4cd41e38167393b6dff0ba22b28b4bb450...,male,16325,IN,4.030312,4.130625,48000,6.240,test,2.0,NaN,1,608,2903,0,NaN,58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7382,common_voice_en_26830417.mp3,ffd4640d6c3d4036401042ff95578afda238716c696695...,female,4851,WIDE,2.745437,3.086125,32000,6.048,test,2.0,NaN,0,142,7277,0,NaN,670
7383,common_voice_en_26830417.mp3,ffd4640d6c3d4036401042ff95578afda238716c696695...,female,4851,THE,1.002000,1.122187,32000,6.048,test,2.0,NaN,0,142,5848,0,NaN,6
7384,common_voice_en_26830417.mp3,ffd4640d6c3d4036401042ff95578afda238716c696695...,female,4851,OF,3.667312,3.727438,32000,6.048,test,2.0,NaN,0,142,4231,0,NaN,46
7385,common_voice_en_26830417.mp3,ffd4640d6c3d4036401042ff95578afda238716c696695...,female,4851,HAS,1.863687,2.024000,32000,6.048,test,2.0,NaN,0,142,2262,0,NaN,9


In [4]:
cv_eval_manifest['target_dur_in_s'] = cv_eval_manifest.end_in_s - cv_eval_manifest.start_in_s

### cut df for wanted restrictions 


In [10]:
(win_size - cv_eval_manifest.target_dur_in_s) / 2 

0       1.309531
1       1.289469
2       1.419750
3       1.299656
4       1.449844
          ...   
7382    1.329656
7383    1.439906
7384    1.469937
7385    1.419844
7386    1.239500
Name: target_dur_in_s, Length: 7387, dtype: float64

In [53]:
np.random.seed(5)
win_size = 3

pad_size = (win_size - cv_eval_manifest.target_dur_in_s) / 2 
# ensure start + pad is enough for 3 second window
valid_dur_egs = cv_eval_manifest[(cv_eval_manifest.start_in_s - pad_size >= 0) & (cv_eval_manifest.end_in_s + pad_size <= cv_eval_manifest.total_file_duration_in_s)]

# ensure end + pad fits in file duration
# valid_dur_egs = valid_dur_egs[valid_dur_egs.end_in_s + pad_size <= valid_dur_egs.total_file_duration_in_s]

# get words long enough for easy comprehension 
valid_dur_valid_char_len = valid_dur_egs[valid_dur_egs.word.str.len() >=5]

female_vocab = valid_dur_valid_char_len[valid_dur_valid_char_len.gender == 'female'].word.unique()
male_vocab = valid_dur_valid_char_len[valid_dur_valid_char_len.gender == 'male'].word.unique()

olap_words = np.intersect1d(male_vocab, female_vocab)

unique_female_vocab = female_vocab[~np.isin(female_vocab, olap_words)]
unique_male_vocab = male_vocab[~np.isin(male_vocab, olap_words)]

n_examp_diff = np.abs(len(unique_female_vocab) - len(unique_male_vocab))

#add n_diff to the smaller set -  female_vocab here
for_female = olap_words[:n_examp_diff]
remaining_olap = olap_words[n_examp_diff:]

unique_female_vocab = np.append(unique_female_vocab, for_female)
print(len(unique_female_vocab), len(unique_male_vocab))


# assign remaining olap words to female & male using weights to ensure gender balance
for_olap = valid_dur_valid_char_len[valid_dur_valid_char_len.gender != 'other']
gend_bal_olap_words = for_olap[for_olap.word.isin(remaining_olap)].groupby(['gender', 'word']).sample(1).groupby('word').sample(1)

female_words = valid_dur_valid_char_len[(valid_dur_valid_char_len.gender == 'female') & (valid_dur_valid_char_len.word.isin(unique_female_vocab))].groupby('word').sample(1)
male_words = valid_dur_valid_char_len[(valid_dur_valid_char_len.gender == 'male') & (valid_dur_valid_char_len.word.isin(unique_male_vocab))].groupby('word').sample(1)

print(female_words.gender.value_counts())
print(male_words.gender.value_counts())
print(gend_bal_olap_words.gender.value_counts())


eval_df = pd.concat([gend_bal_olap_words, female_words, male_words]).reset_index()
print(eval_df.gender.value_counts())

eval_df


265 265
female    265
Name: gender, dtype: int64
male    265
Name: gender, dtype: int64
female    35
male      33
Name: gender, dtype: int64
female    300
male      298
Name: gender, dtype: int64


,index,aud_path,client_id,gender,origin_index,word,start_in_s,end_in_s,sr,total_file_duration_in_s,dataset_splits,dataset_split_int,wsn_word_int,gender_int,client_int,word_examp_ix,cv_talker_int,cv_speaker_ident_split,cv_word_int,target_dur_in_s
0,1130,common_voice_en_23615248.mp3,1181743c491f01375fb804dac1fe93f37b6c8d2a914d0a...,female,18467,QUICKLY,2.642812,3.103312,48000,7.512,test,2.0,565.0,0,305,4784,0,NaN,541,0.460500
1,6618,common_voice_en_31266632.mp3,d71480091dd972c0ea669f5f3b03ca979254d14a859154...,female,14486,RAILWAY,3.408000,3.708687,32000,5.148,test,2.0,NaN,0,908,4790,0,NaN,440,0.300687
2,1919,common_voice_en_21773347.mp3,1f6d85ba2b1ac13901b12419e30f9f9bdd3d0f66a54bf1...,male,593,RANGE,4.965438,5.185688,48000,7.272,test,2.0,569.0,1,1333,4792,0,NaN,485,0.220250
3,4461,common_voice_en_13714943.mp3,7546b3fb3a17558658d3d6e81a487d0786bd853c1acd08...,male,2193,RELEASE,1.503937,2.025313,48000,6.120,test,2.0,590.0,1,1345,4831,0,NaN,516,0.521375
4,4323,common_voice_en_27846447.mp3,721617406788741a3bc96aba412d478fb4ec2d274075ef...,male,10011,RENAMED,4.032563,4.373625,32000,7.740,test,2.0,NaN,1,733,4838,0,NaN,951,0.341062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593,1614,common_voice_en_19813863.mp3,19149de6773e2e113c32363a2b457def915a4dc8eaff3d...,male,19295,WIDELY,2.165437,2.566438,48000,7.944,test,2.0,774.0,1,887,7280,0,NaN,576,0.401000
594,4706,common_voice_en_17417536.mp3,7ee0a4336692119bff031932ea475966132b7202b00d73...,male,2377,WITHOUT,1.945000,2.245812,48000,4.656,test,2.0,778.0,1,1440,7380,0,NaN,261,0.300812
595,5287,common_voice_en_22727683.mp3,984aa6fd73e81d038797771d2dd1831e7070fe44664a54...,male,11719,WOMEN,4.227875,4.628625,48000,6.456,test,2.0,779.0,1,288,7386,0,NaN,492,0.400750
596,402,common_voice_en_21401015.mp3,05ec60dfed3e6fe3781634fb751ff69293370980db6519...,male,5150,WRITING,3.753250,4.094500,48000,5.664,test,2.0,788.0,1,497,7437,0,NaN,552,0.341250


In [54]:
eval_df.to_pickle(commonvoice_path / 'manifests/cv_model_and_participant_eval_manifest.pdpkl')

In [55]:
word_dict = eval_df.word.str.lower().unique()
word_dict = dict(dictionary=list(np.sort(word_dict)))
print(len(word_dict))

# save as json for jspsych compat
import json

out_name = commonvoice_path / 'cv_model_and_participant_eval_word_dict.json'

with open(out_name, 'w', encoding='utf-8') as f:
    json.dump(word_dict, f, ensure_ascii=False, indent=4)





1


In [56]:
# # Take same number of males as females
# np.random.seed(0)

# # get number of examples for gender classes
# samples_per_gender = {gender:count for gender,count in cv_eval_manifest.gender.value_counts().items()}
# # more males than females. Take all N female examples and only N male examples
# samples_per_gender['male'] = samples_per_gender['female'] 
# # show samples to draw per gender group
# del samples_per_gender['other']
# print(samples_per_gender)
# # sample from all data
# to_use = cv_eval_manifest[cv_eval_manifest.gender.isin(['male', 'female'])]
# gend_bal_new = to_use.groupby('gender').apply(lambda group: group.sample(samples_per_gender[group.name]))
# gend_bal_new = gend_bal_new.reset_index(drop=True)


In [57]:
eval_df.iloc[0]

index                                                                    1130
aud_path                                         common_voice_en_23615248.mp3
client_id                   1181743c491f01375fb804dac1fe93f37b6c8d2a914d0a...
gender                                                                 female
origin_index                                                            18467
word                                                                  QUICKLY
start_in_s                                                           2.642812
end_in_s                                                             3.103312
sr                                                                      48000
total_file_duration_in_s                                                7.512
dataset_splits                                                           test
dataset_split_int                                                         2.0
wsn_word_int                                                    

In [58]:
import sys
sys.path.append('/om2/user/imgriff/datasets/spatial_audio_pipeline/utils')
import h5_from_pandas

In [59]:
cv_aud_path = Path('/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27/en/clips')
short_ixs = []
for ix in range(len(eval_df)):
    eg = eval_df.iloc[ix]
    aud, on, offset = h5_from_pandas.get_excerpt(cv_aud_path/eg.aud_path, eg.sr, eg.start_in_s, eg.end_in_s, eg.total_file_duration_in_s)
    if aud is None:
        print("None on ix ", ix)
        short_ixs.append(ix)
# raw_aud

### Make sure h5 worked

In [67]:
import h5py 
from IPython.display import Audio


In [71]:
cv_class_map = {v:k for k,v in cv_vocab.items()}

In [62]:
h5_path = '/om2/user/imgriff/datasets/commonvoice_9_en/3000ms/stimSR_50000/cv_9_en/subsets/model_and_participant_test_set/model_and_participant_test_set_50000Hz_rate_60dB_000.hdf5'

h5 = h5py.File(h5_path, 'r')

In [68]:
h5.keys()

<KeysViewHDF5 ['client_int', 'dataset_split', 'index_file', 'index_manifest', 'label_talker_int', 'label_talker_sex_int', 'label_word_int', 'parent_timestamp_signal_end', 'parent_timestamp_signal_start', 'parent_timestamp_target_end', 'parent_timestamp_target_start', 'signal', 'sr']>

In [73]:

eg_ix = 10

print(cv_class_map[h5['label_word_int'][eg_ix]])
Audio(h5['signal'][eg_ix], rate=h5['sr'][eg_ix])

recently
